In [1]:
import os
import glob
import numpy as np
import pandas as pd
import nibabel as nib
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

c:\Users\juan_\AppData\Local\Programs\Python\Python310\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.25.905) or chardet (7.6.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [3]:
# 1. Define custom PyTorch Dataset for Axial Slices
class COBRE_AxialDataset(Dataset):
    def __init__(self, subject_paths, labels, img_size=128):
        self.subject_paths = subject_paths
        self.labels = labels
        self.img_size = img_size
        
    def __len__(self):
        return len(self.subject_paths)
        
    def __getitem__(self, idx):
        img_path = self.subject_paths[idx]
        
        # Load 3D volume via nibabel
        nii_img = nib.load(img_path)
        volume = nii_img.get_fdata()
        
        # Extract central axial slice
        mid_slice_idx = volume.shape[2] // 2
        slice_2d = volume[:, :, mid_slice_idx]
        
        # Normalize intensity to [0, 1] range safely
        min_val, max_val = np.min(slice_2d), np.max(slice_2d)
        if max_val > min_val:
            slice_2d = (slice_2d - min_val) / (max_val - min_val)
        else:
            slice_2d = np.zeros_like(slice_2d)
            
        # Resize/pad to standard square dimensions (e.g., 128x128) using PyTorch interpolation
        tensor_slice = torch.tensor(slice_2d, dtype=torch.float32).unsqueeze(0).unsqueeze(0) # Shape: [1, 1, H, W]
        resized_slice = nn.functional.interpolate(tensor_slice, size=(self.img_size, self.img_size), mode='bilinear', align_corners=False)
        
        # Remove batch/channel extra dims to return [1, H, W]
        x = resized_slice.squeeze(0)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        
        return x, y

In [4]:
# 2. Prepare data mapping
data_dir = r"C:\Users\juan_\Desktop\cv-medical\COBRE_Dataset"
csv_path = os.path.join(data_dir, "participants.tsv")
df_participants = pd.read_csv(csv_path, sep='\t')

In [5]:
# Match available files with clinical labels
all_paths = sorted(glob.glob(os.path.join(data_dir, "sub-*", "anat", "*_T1w.nii.gz")))
path_dict = {os.path.basename(p).split('_')[0]: p for p in all_paths}

valid_participants = df_participants[df_participants['participant_id'].isin(path_dict.keys())].copy()
matched_paths = [path_dict[pid] for pid in valid_participants['participant_id']]

In [6]:
# Encode string labels (CONTROL, SCHZ, etc.) to integers
encoder = LabelEncoder()
encoded_labels = encoder.fit_transform(valid_participants['diagnosis'])
num_classes = len(encoder.classes_)

print(f"Classes detected: {list(encoder.classes_)}")
print(f"Total samples matched for Deep Learning: {len(matched_paths)}")

Classes detected: ['ADHD', 'BIPOLAR', 'CONTROL', 'SCHZ']
Total samples matched for Deep Learning: 265


In [7]:
# 3. Train/Val Split
X_train_p, X_val_p, y_train_l, y_val_l = train_test_split(
    matched_paths, encoded_labels, test_size=0.2, random_state=42, stratify=encoded_labels
)

train_dataset = COBRE_AxialDataset(X_train_p, y_train_l)
val_dataset = COBRE_AxialDataset(X_val_p, y_val_l)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)